---
---
# Loading Libraries & Dataframe
---
---

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import math

from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest, RandomForestClassifier, VotingClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, TargetEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import (classification_report, roc_curve, auc,
                                     precision_score, recall_score, f1_score, fbeta_score, make_scorer, precision_recall_curve)
from sklearn.compose import ColumnTransformer
from sklearn.base   import BaseEstimator, TransformerMixin, clone
from sklearn.calibration import CalibratedClassifierCV

from imblearn.pipeline import Pipeline

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

In [ ]:
path = kagglehub.dataset_download("vivekvivek13/bank-customers-prediction")
df = pd.read_csv(f"{path}/bank.csv", delimiter=';')
df

In [ ]:
df.info()

---
---
# Preprocessing
---
---

## Data_Type Function
---

In [ ]:
# Shows the data type of each column (int, float, object...)
print(f"Data Type Inspection:\n{df.dtypes}")

default : object -> Bool

housing : object -> Bool

loan    : object -> Bool

month   : object -> int OR string

day_of_week : object -> int OR string

## Unknown Counts Report
---

In [ ]:
# Counts how many 'unknown' values are in each column
unknown_counts = (df == 'unknown').sum()
unknown_pct = (unknown_counts / len(df)) * 100

summary = pd.DataFrame({
    'Unknown Count': unknown_counts,
    'Percentage (%)': unknown_pct.round(2)
})
summary[summary['Unknown Count'] > 0].sort_values(by='Unknown Count', ascending=False)

## Unique Values Report

---



In [ ]:
unique_data = []
for col in df.columns:
    uniques = df[col].unique()
    unique_data.append({
            'Column Name': col,
            'Unique Count': len(uniques),
            'Sample Values': list(uniques[:15])
        })

report_df = pd.DataFrame(unique_data)

print(f"{'='*20} Unique Values Report: {'='*20}")
report_df

## Job & Marital & Education
---

In [ ]:
df['job'].unique()

In [ ]:
df['education'].unique()

In [ ]:
df['marital'].unique()

In [ ]:
# Dictionary containing concise English definitions for each job
job_definitions = {
        'housemaid': 'Responsible for domestic tasks like cleaning and cooking.',
        'services': 'Jobs in the service sector (e.g., retail, waiters, security).',
        'admin.': 'Office administrators and clerical staff.',
        'blue-collar': 'Manual workers in industries like construction or manufacturing.',
        'technician': 'Specialized technical staff (e.g., IT, engineering support).',
        'retired': 'People who have stopped working permanently due to age.',
        'management': 'Executives and department managers with decision-making roles.',
        'unemployed': 'Individuals who are currently not working.',
        'self-employed': 'Individuals working for themselves (freelancers/contractors).',
        'unknown': 'Missing information regarding the client\'s occupation.',
        'entrepreneur': 'Self-employed individuals running their own business ventures.',
        'student': 'People currently enrolled in schools or universities.'
    }

# Identifying unique values present in the 'job' column
unique_jobs = df['job'].unique()

print(f"\n{'='*100}")
print(f"{'JOB CATEGORY':<18} | {'ROLE DESCRIPTION'}")
print(f"{'='*100}")

# Iterating and printing each job with its meaning
for job in sorted(unique_jobs):
    meaning = job_definitions.get(job, "Definition not found.")
    print(f"{job.capitalize():<18} | {meaning}")
    print(f"{'-'*100}")

In [ ]:
def standardize_text_columns(df):

    # 1. Clean 'job' column: Replace '-' and '.' with spaces
    if 'job' in df.columns:
        df['job'] = df['job'].str.replace('-', ' ', regex=False).str.replace('.', ' ', regex=False).str.strip()

    # 2. Clean 'marital' column: Rename 'divorced' to include widowed reference
    if 'marital' in df.columns:
        df['marital'] = df['marital'].replace('divorced', 'divorced & widowed')

    # 3. Clean 'education' column: Map values to the requested school levels
    education_mapping = {
        'basic.4y': 'primary school',
        'basic.6y': 'middle school',
        'basic.9y': 'secondary school',
        'high.school': 'high school',
        'professional.course': 'professional course',
        'university.degree': 'university degree',
        'illiterate': 'illiterate',
        'unknown': 'unknown'
    }

    if 'education' in df.columns:
        df['education'] = df['education'].replace(education_mapping)

    print("--- Cleaning of Job, Marital, and Education columns is complete. ---")
    return df

df = standardize_text_columns(df)

## Month & Day of Weak  Function
---

In [ ]:
df['month'].unique()

In [ ]:
df['day_of_week'].unique()

In [ ]:
def encode_temporal_columns(df):
    # Convert month names to numbers (Jan=1 → Dec=12)
    month_map = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }

    # Convert days to numbers (Mon=1 → Sun=7)
    day_map = {
        'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4,
        'fri': 5, 'sat': 6, 'sun': 7
    }

    # Apply mapping on 'month' column
    if 'month' in df.columns:
        df['month'] = df['month'].str.lower().map(month_map)

    # Apply mapping on 'day_of_week' column
    if 'day_of_week' in df.columns:
        df['day_of_week'] = df['day_of_week'].str.lower().map(day_map)

    print("--- Encoding for 'month' and 'day_of_week' is complete ---")
    return df

# Apply the function
df = encode_temporal_columns(df)
df

## Pdays Column
---

In [ ]:
np.sort(df['pdays'].unique())

In [ ]:
def transform_pdays(df):
    # create new column
    df['pdays_group'] = df['pdays']

    df.loc[df['pdays'] == 999, 'pdays_group'] = 'not contacted'

    mask = df['pdays'] != 999

    df.loc[mask, 'pdays_group'] = pd.cut(
        df.loc[mask, 'pdays'],
        bins=[-1, 3, 7, 14, 30],
        labels=['last 3 days','last week','last 2 weeks','last month']
    )
    return df

df = transform_pdays(df)

In [ ]:
df['pdays_group'].unique()

In [ ]:
def get_pdays_summary(df):

    # Create a temporary series to hold labels
    temp_series = df['pdays'].astype(object)

    # 1. Identify and label the contacted group
    mask = df['pdays'] != 999
    temp_series.loc[mask] = pd.cut(
        df.loc[mask, 'pdays'],
        bins=[-1, 3, 7, 14, 30],
        labels=['last 3 days', 'last week', 'last 2 weeks', 'last month']
    )

    # 2. Label the 'not contacted' group
    temp_series.loc[df['pdays'] == 999] = 'not contacted'

    # 3. Create the summary table
    summary = temp_series.value_counts().reset_index()
    summary.columns = ['Bin Label', 'Count']
    summary['Percentage (%)'] = (summary['Count'] / len(df) * 100).round(2)

    return summary

get_pdays_summary(df)

## Age
---

In [ ]:
# Age binning rule:
# > 60  -> adult
# > 21  -> teens
# else  -> senior

df['age'] = np.select(
    [
        df['age'] > 60,
        df['age'] > 21
    ],
    [
        'adult',
        'teen'
    ],
    default='senior'
)

df[['age']].head()





## Outlires Dettection
---

In [ ]:
def detect_outliers(df, show_plots=True):

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    summary_data = []

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        count = len(outliers)

        summary_data.append({
            'Column': col,
            'Outlier Count': count,
            'Percentage (%)': round((count / len(df)) * 100, 2)
        })

    if show_plots:
        num_cols = len(numeric_cols)
        fig, axes = plt.subplots(1, num_cols, figsize=(num_cols * 4, 5))
        if num_cols == 1: axes = [axes]
        for i, col in enumerate(numeric_cols):
            sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
            axes[i].set_title(f'Manual Check: {col}')
        plt.tight_layout()
        plt.show()

    return pd.DataFrame(summary_data)

detect_outliers(df)

##  IsolationForest


In [ ]:
def handle_outliers_isolation_forest(df, contamination = 0.05):

    df_cleaned = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if numeric_cols.empty:
        print("No numerical columns found.")
        return df_cleaned


    iso = IsolationForest(contamination=contamination, random_state=42)


    data_for_model = df[numeric_cols].fillna(df[numeric_cols].median())
    preds = iso.fit_predict(data_for_model)

    outlier_indices = df.index[preds == -1]
    df_cleaned = df_cleaned.drop(index=outlier_indices)

    print("\n----- Isolation Forest Summary -----")
    print(f"Total Rows Before: {len(df)}")
    print(f"Detected and Removed: {len(outlier_indices)} rows ({(len(outlier_indices)/len(df)*100):.2f}%)")
    print(f"Total Rows After: {len(df_cleaned)}")
    print("-" * 36)

    return df_cleaned

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['duration'], kde=True, color='teal', bins=50)
plt.title('Distribution of Call Duration', fontsize=15)
plt.xlabel('Duration (seconds)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.show()

### After Islation Forest

In [ ]:
detect_outliers(df)

## Imputation
---

In [ ]:
def plot_unknown_values(df):

    unknown_counts = (df == 'unknown').sum()
    unknown_data = unknown_counts[unknown_counts > 0].sort_values(ascending=False)
    if unknown_data.empty:
        print("No 'unknown' values found in the dataset.")
        return
    unknown_pct = (unknown_data / len(df)) * 100
    plt.figure(figsize=(12, 6))
    sns.barplot(x=unknown_pct.values, y=unknown_pct.index, palette='Reds_r')
    for i, v in enumerate(unknown_pct.values):
        plt.text(v + 0.5, i, f'{v:.2f}%', color='black', va='center')
    plt.title('Percentage of "Unknown" Values per Column', fontsize=15)
    plt.xlabel('Percentage (%)', fontsize=12)
    plt.ylabel('Columns', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

    summary = pd.DataFrame({'Count': unknown_data, 'Percentage (%)': unknown_pct.round(2)})
    return summary

plot_unknown_values(df)

In [ ]:
# 1. Convert 'unknown' strings to actual NaN values so the library can detect them
df.replace('unknown', np.nan, inplace=True)

# ==========================================
# Chart 1: Bar Chart
# Shows the total count of non-missing values per column
# ==========================================
msno.bar(df, figsize=(12, 6), color="dodgerblue", fontsize=12)
plt.title('Count of Complete Data per Column', fontsize=16)
plt.show()

# ==========================================
# Chart 2: Matrix Plot
# Excellent for seeing WHERE the missing values are and if they are related
# ==========================================
msno.matrix(df, figsize=(12, 6), color=(0.2, 0.4, 0.6), sparkline=False)
plt.title('Missing Data Map (White lines indicate missing values)', fontsize=16)
plt.show()

In [ ]:
class CategoricalKNNImputer(BaseEstimator, TransformerMixin):
    """
    Ordinally encodes categoricals → KNN imputes → rounds back → decodes.
    Handles numeric columns transparently alongside categoricals.
    """

    def __init__(self, n_neighbors=5, weights="uniform"):
        self.n_neighbors = n_neighbors
        self.weights     = weights

    def fit(self, X, y=None):
        df = pd.DataFrame(X).copy()

        self.cat_cols_ = df.select_dtypes(include=["object", "category"]).columns.tolist()

        self.mappings_         = {}   # str to int
        self.reverse_mappings_ = {}   # int to str

        for col in self.cat_cols_:
            unique_vals = df[col].dropna().unique()
            self.mappings_[col]         = {val: i for i, val in enumerate(unique_vals)}
            self.reverse_mappings_[col] = {i: val for i, val in enumerate(unique_vals)}

        df = self._encode(df)
        self.imputer_ = KNNImputer(
            n_neighbors = self.n_neighbors,
            weights     = self.weights,
            metric      = "nan_euclidean"
        )
        self.imputer_.fit(df)

        return self 

    def transform(self, X, y=None):
        df = pd.DataFrame(X).copy()
        df      = self._encode(df)
        arr     = self.imputer_.transform(df)
        df[:]   = arr

        for col in self.cat_cols_:
            col_idx        = df.columns.get_loc(col)
            max_idx        = len(self.reverse_mappings_[col]) - 1
            df[col]        = np.round(df[col])
            df[col]        = np.clip(df[col], 0, max_idx)
            df[col]        = df[col].map(self.reverse_mappings_[col])

        return df

    def _encode(self, df):
        for col in self.cat_cols_:
            df[col] = df[col].map(self.mappings_[col])
        return df

In [ ]:
plot_unknown_values(df)

---
---
# EDA
---
---

## Information Dataset
---

In [ ]:
train_df, test_df = train_test_split(
    df, test_size = .2,
    random_state = 30, stratify = df['y']
)

if train_df['y'].dtype == 'object':
    # .replace ensures values not in {'yes', 'no'} aren't nuked to NaN
    train_df['y'] = train_df['y'].replace({'no': 0, 'yes': 1}).astype(int)
    test_df['y'] = test_df['y'].replace({'no': 0, 'yes': 1}).astype(int)

In [ ]:
train_df.shape

In [ ]:
train_df.describe(include = 'all')

## Charts
---

### Visualizing the Target Variable Distribution (Class Balance)
The very first step in any classification project is to check the distribution of your target variable (`y`). We use a simple **Countplot** (bar chart) to see how many instances of 'yes' and 'no' exist.

In [ ]:
sns.countplot(x = 'y',data = train_df)
plt.title('Target Variable Distribution (y)')
plt.show()

### Visualizing Numerical Features and Outliers vs Target Variable
To understand how numerical features vary across the target classes (and to identify potential outliers), we use a grid of **Boxplots**. This allows us to see the median, spread, and extreme values of each feature for both the 'yes' and 'no' outcomes simultaneously.

In [ ]:
num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.drop('y')

plt.figure(figsize=(16, 18))
for i, col in enumerate(num_cols, 1):
    plt.subplot(4, 3, i)
    sns.boxplot(x='y', y=col, data=train_df, palette='Set2')
    plt.title(f'Outliers in {col}', fontsize=12, fontweight='bold')
    plt.xlabel('Target (y)')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

### Visualizing Categorical Features vs Target Variable
To understand the relationship between categorical attributes and the target variable `y`, we use a grid of **Grouped Bar Charts**. This helps identify which categories have a higher conversion rate (e.g., which jobs or education levels are more likely to say "yes").

In [ ]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'pdays_group', 'contact', 'poutcome']

fig, axes = plt.subplots(3, 3, figsize=(15, 20))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(x=col, hue='y', data=train_df, ax=axes[i], palette='Set1')
    axes[i].set_title(f'{col.capitalize()} vs Target (y)', fontsize=14, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45) 
    axes[i].set_ylabel('Count')


plt.tight_layout()
plt.show()

### Visualizing Data Distributions (KDE Plots)
To understand the underlying distribution of our numerical features and see how well they separate our target classes, we use **Kernel Density Estimate (KDE) plots**. A KDE plot is like a smoothed histogram. If the curves for 'yes' and 'no' are heavily overlapping, the feature might not be very useful. If the peaks are separated, the feature is a strong predictor.

In [ ]:
cols_per_row = 2
num_plots = len(num_cols)
num_rows = math.ceil(num_plots / cols_per_row)

fig, axes = plt.subplots(num_rows, cols_per_row, figsize=(15, num_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.kdeplot(
        data=train_df, 
        x=col, 
        hue='y', 
        fill=True, 
        palette='Set2', 
        ax=axes[i]
    )
    axes[i].set_title(f'Distribution of {col} by Target')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')

# Clean up: Remove empty subplots if num_plots is odd
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Visualizing Feature Correlation with the Target
To determine which numerical features have the strongest mathematical relationship with our target variable, we calculate the **Pearson Correlation Coefficient**.

In [ ]:
corr = train_df.corr(numeric_only = True).drop('y')

matrix = corr[['y']].sort_values(by = 'y', ascending = False)

plt.figure(figsize = (4, 8))
sns.heatmap(matrix,
            annot = True,
            fmt = '.2f',
            cmap = 'coolwarm',
            vmin = -1,
            vmax = 1)
plt.title('Correlation of features with target')
plt.tight_layout()
plt.show()

### Dimensionality Reduction and Visualization (t-SNE)
To understand if our features contain natural patterns or clusters that separate the target classes, we use **t-SNE** (t-Distributed Stochastic Neighbor Embedding).

In [ ]:
df_viz = train_df.copy()

# Handle categorical features if any remain
X_viz = pd.get_dummies(df_viz.drop(columns=['y']), drop_first=True)
y_viz = df_viz['y']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_viz)

tsne = TSNE(n_components=2, 
            perplexity=30,
            max_iter=1000, 
            random_state=42, 
            init='pca', 
            learning_rate='auto')

X_tsne = tsne.fit_transform(X_scaled)

# Visualization
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=X_tsne[:, 0], 
    y=X_tsne[:, 1], 
    hue=y_viz, 
    palette='Set2', 
    alpha=0.6, 
    s=20,
    edgecolor='none'
)

plt.title('t-SNE Visualization of Feature Space')
plt.xlabel('t-SNE component 1')
plt.ylabel('t-SNE component 2')
plt.legend(title='Target (y)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
---
# Modeling
---
---

## Feature Engineering
---

In [ ]:
def engineer_features(df):
    temp_df = df.copy()
    
    temp_df['was_prev_success'] = (temp_df['poutcome'] == 'success').astype(int)
    temp_df['econ_cycle'] = temp_df['euribor3m'] * temp_df['emp.var.rate']
    temp_df['low_euribor'] = (temp_df['euribor3m'] < 2).astype(int)
    temp_df['low_employment'] = (temp_df['nr.employed'] < 5100).astype(int)
    temp_df['recession_signal'] = temp_df['low_euribor'] * temp_df['low_employment']
    temp_df['over_contacted'] = (temp_df['campaign'] > 5).astype(int)
    temp_df['is_new_client'] = (temp_df['previous'] == 0).astype(int)
    temp_df['total_interactions'] = temp_df['campaign'] + temp_df['previous']
    temp_df['high_stability'] = ((temp_df['default'] == 'no') & 
                                 (temp_df['housing'] == 'yes') & 
                                 (temp_df['loan'] == 'no')).astype(int)
    temp_df['macro_stress'] = temp_df['emp.var.rate'] / (temp_df['nr.employed'] + 1e-6)
    temp_df['peak_months'] = temp_df['month'].isin([5, 6, 8]).astype(int)

    return temp_df

In [ ]:
train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

## Splitting/Encoding/Scaling
---

In [ ]:
train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

drop_cols = ['y', 'duration', 'pdays','day_of_week', 'month', 'cons.conf.idx', 'emp.var.rate',
             'campaign', 'previous', 'low_euribor', 'low_employment', 'nr.employed']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['y']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['y']

ohe_cols = ['marital', 'housing', 'loan', 'poutcome', 'pdays_group', 'default', 'contact', 
            'was_prev_success', 'recession_signal', 'age',
            'over_contacted', 'is_new_client', 'high_stability', 'peak_months']

te_cols = ['job', 'education']
scale_cols = ['euribor3m', 'econ_cycle',
              'total_interactions', 'macro_stress']

imputer = CategoricalKNNImputer(n_neighbors=5, weights='uniform')

X_train= pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)
X_test= pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")
print(f"Class distribution : {np.bincount(y_test)}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('te', TargetEncoder(smooth = 'auto', cv = 5), te_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols),
        ('num', RobustScaler(), scale_cols)
    ], remainder = 'passthrough')

print('Encodng & Scaling Complete')

In [ ]:
corr = train_df.corr(numeric_only = True).drop(['y', 'euribor3m', 'emp.var.rate', 'duration', 'pdays'])

matrix = corr[['y']].sort_values(by = 'y', ascending = False)

plt.figure(figsize = (4, 8))
sns.heatmap(matrix,
            annot = True,
            fmt = '.2f',
            cmap = 'coolwarm',
            vmin = -1,
            vmax = 1)
plt.title('Correlation of features with target')
plt.tight_layout()
plt.show()

## Checking Multicollineairty
---

In [ ]:
X_numeric = X_train.select_dtypes(include=[np.number])
X_vif_input = add_constant(X_numeric)

# Build the VIF DataFrame
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif_input.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_input.values, i) for i in range(X_vif_input.shape[1])]

# Sort and display
vif_results = vif_data[vif_data["Feature"] != 'const'].sort_values(by="VIF", ascending=False)
print(vif_results)

## Setting up model variables
---

In [ ]:
# CONFIG
random_state = 30
n_splits     = 5
n_trials     = 50
scoring      = "recall"
thresholds = [.2, .25, .3, .35, .4, .45, .5]
results    = []
f2_scorer = make_scorer(fbeta_score, beta=2)

skf = StratifiedKFold(n_splits = n_splits, shuffle = True, random_state = random_state)

# Each model block writes its proba here
test_proba = {}

## Logistic Regression
---

In [ ]:
# Optuna objective
def objective_lr(trial):
    params = {
        "C"           : trial.suggest_float("C", 0.001, 10.0, log = True),
        "max_iter"    : trial.suggest_int("max_iter", 200, 1000),
        "class_weight": trial.suggest_categorical("class_weight", [None, 'balanced']),
    }
    model = LogisticRegression(**params, random_state = 30)
    
    pipeline = Pipeline([
        ('scaler/encoder', preprocessor),
        ('classifier', model)
    ])
    
    # CV is run on the already-scaled train set
    scores = cross_val_score(pipeline, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()
    return scores.mean() - (scores.std() * .5)

study_lr = optuna.create_study(
    direction="maximize",
    sampler = optuna.samplers.TPESampler(seed = random_state)
)
study_lr.optimize(objective_lr, n_trials= n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_lr.best_value:.3f}")
print(f"  Best params      : {study_lr.best_params}")

# Refit on full train set with best params
lr_best = pipeline = Pipeline([
        ('scaler/encoder', clone(preprocessor)),
        ('classifier', LogisticRegression(**study_lr.best_params, random_state = random_state))
    ])
lr_best.fit(X_train, y_train)

lr_proba = lr_best.predict_proba(X_test)[:, 1]
lr_pred = lr_best.predict(X_test)
test_proba["Logistic Regression"] = lr_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, lr_pred, digits = 2))
print("-"*55)

# Assuming a €5 cost per call attempt
cost_per_call = 5 
precision_class_1 = classification_report(y_test, lr_pred, output_dict = True)['1']['precision']
impact = cost_per_call / precision_class_1 if precision_class_1 > 0 else float('inf')

print(f"Economic Impact: €{impact:.2f} per subscriber")
print(f"Target Met: {impact < 35}")

In [ ]:
y_train_probas = cross_val_predict(lr_best, X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

# Calculate threshold metrics
precisions, recalls, thresholds_pr = precision_recall_curve(y_train, y_train_probas)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)

# Applying target metric constraints
mask = (recalls[:-1] > 0.75) & (f1_scores[:-1] > 0.55)

if not any(mask):
    print("❌ Targets Unreachable: No threshold satisfies both Recall > 0.75 and F1 > 0.55.")
    # Fallback: Find the threshold that gets closest (Maximizing the product of both)
    best_idx = np.argmax(recalls[:-1] * f1_scores[:-1])
    print("Finding the best compromise instead...")
else:
    # Within the successful thresholds, find the one with the highest F1 to protect the budget
    valid_indices = np.where(mask)[0]
    best_idx = valid_indices[np.argmax(f1_scores[valid_indices])]
    print("✅ Targets Reached on training data.")

optimal_threshold = thresholds_pr[best_idx]
print(f"✔ Target-Driven Threshold: {optimal_threshold:.4f}")

# Final Evaluation
lr_proba_test = lr_best.predict_proba(X_test)[:, 1]
lr_preds_final = (lr_proba_test >= optimal_threshold).astype(int)

print("\n" + "-"*55)
print(" FINAL TEST EVALUATION (Target: Recall > 0.75, F1 > 0.55)")
print("-"*55)
print(classification_report(y_test, lr_preds_final, digits=4))

# Economic Impact Check
prec_final = precision_score(y_test, lr_preds_final)
impact = 5 / prec_final if prec_final > 0 else float('inf')
print(f"Economic Impact: €{impact:.2f} | Target Met: {impact < 35}")

## XGBoost Classifier
---

In [ ]:
# Optuna objective
def objective_xgb(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 200, 800),
        "max_depth"       : trial.suggest_int("max_depth", 3, 7),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 0.0001, 10.0, log = True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 0.0001, 10.0, log = True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0)
    }    
    model = XGBClassifier(
        **params, random_state = random_state,
        eval_metric="logloss", verbosity=0, n_jobs=-1
    )
    
    pipeline = Pipeline([
        ('scaler/encoder', clone(preprocessor)),
        ('classifier', model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()
    
    return scores.mean() - (scores.std() * .5)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed = random_state)
)
study_xgb.optimize(objective_xgb, n_trials = n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_xgb.best_value:.2f}")
print(f"  Best params      : {study_xgb.best_params}")

# Refit on full train set with best params
xgb_best = pipeline = Pipeline([
        ('scaler/encoder', preprocessor),
        ('classifier', XGBClassifier(**study_xgb.best_params, random_state = random_state, eval_metric="logloss", verbosity=0, n_jobs=-1))
    ])
xgb_best.fit(X_train, y_train)
xgb_pred = xgb_best.predict(X_test)
xgb_proba = xgb_best.predict_proba(X_test)[:, 1]

test_proba["XGBoost"] = xgb_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, xgb_pred, digits = 2))
print("-"*55)

# Assuming a €5 cost per call attempt
cost_per_call = 5
precision_class_1 = classification_report(y_test, xgb_pred, output_dict = True)['1']['precision']
impact = cost_per_call / precision_class_1 if precision_class_1 > 0 else float('inf')

print(f"Economic Impact: €{impact:.2f} per subscriber")
print(f"Target Met: {impact < 35}")

In [ ]:
y_train_probas_xgb = cross_val_predict(xgb_best, X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

# Calculate threshold metrics
prec_xgb, rec_xgb, thresh_xgb = precision_recall_curve(y_train, y_train_probas_xgb)
f1_xgb = (2 * prec_xgb * rec_xgb) / (prec_xgb + rec_xgb + 1e-9)

# Applying target metric constraints
mask_xgb = (rec_xgb[:-1] > 0.75) & (f1_xgb[:-1] > 0.55)

if not any(mask_xgb):
    print("❌ Targets Unreachable with current features: Finding best compromise...")
    best_idx_xgb = np.argmax(rec_xgb[:-1] * f1_xgb[:-1])
else:
    print("✅ Targets Reached on training data folds.")
    valid_indices = np.where(mask_xgb)[0]
    best_idx_xgb = valid_indices[np.argmax(f1_xgb[valid_indices])]

optimal_threshold_xgb = thresh_xgb[best_idx_xgb]

# Final evaluation
xgb_proba_test = xgb_best.predict_proba(X_test)[:, 1]
xgb_preds_final = (xgb_proba_test >= optimal_threshold_xgb).astype(int)
test_proba["XGBoost"] = xgb_proba_test # Save for comparison

print("\n" + "-"*55)
print(f" XGBOOST FINAL REPORT (Threshold: {optimal_threshold_xgb:.4f})")
print("-"*55)
print(classification_report(y_test, xgb_preds_final, digits=4))

# Economic Impact Check
prec_final = precision_score(y_test, xgb_preds_final)
impact = 5 / prec_final if prec_final > 0 else float('inf')
print(f"Economic Impact: €{impact:.2f} | Target Met: {impact < 35}")

## LightGBM Classifier
---

In [ ]:
# Optuna objective
def objective_lgbm(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"       : trial.suggest_int("max_depth", 3, 12),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.001, 0.3, log = True),
        "num_leaves"      : trial.suggest_int("num_leaves", 20, 150),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 0.0001, 10.0, log = True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 0.0001, 10.0, log = True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0)
    }
    model = LGBMClassifier(**params, random_state = random_state, n_jobs=-1, verbose=-1)
    
    pipeline = Pipeline([
        ('scaler/encoder', clone(preprocessor)),
        ('classifier', model)
    ])
    
    scores = cross_val_score(pipeline, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()
    
    return scores.mean() -  (scores.std() * .5)

study_lgbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed = random_state)
)
study_lgbm.optimize(objective_lgbm, n_trials = n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_lgbm.best_value:.3f}")
print(f"  Best params      : {study_lgbm.best_params}")

# Refit on full train set with best params
lgbm_best = pipeline = Pipeline([
        ('scaler/encoder', preprocessor),
        ('classifier', LGBMClassifier(**study_lgbm.best_params, random_state = random_state, n_jobs=-1, verbose=-1))
    ])
lgbm_best.fit(X_train, y_train)

lgbm_proba = lgbm_best.predict_proba(X_test)[:, 1]
lgbm_pred = lgbm_best.predict(X_test)
test_proba["LightGBM"] = lgbm_proba   # save for ROC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, lgbm_pred, digits = 2))
print("-"*55)

# Assuming a €5 cost per call attempt
cost_per_call = 5 
precision_class_1 = classification_report(y_test, lgbm_pred, output_dict = True)['1']['precision']
impact = cost_per_call / precision_class_1 if precision_class_1 > 0 else float('inf')

print(f"Economic Impact: €{impact:.2f} per subscriber")
print(f"Target Met: {impact < 35}")

In [ ]:
y_train_probas_lgbm = cross_val_predict(lgbm_best, X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

# Calculate thresholds
prec_lgbm, rec_lgbm, thresh_lgbm = precision_recall_curve(y_train, y_train_probas_lgbm)
f1_lgbm = (2 * prec_lgbm * rec_lgbm) / (prec_lgbm + rec_lgbm + 1e-9)

# Apply target metric constraints
mask_lgbm = (rec_lgbm[:-1] > 0.75) & (f1_lgbm[:-1] > 0.55)

if not any(mask_lgbm):
    print("❌ Targets Unreachable: Finding best mathematical compromise...")
    best_idx_lgbm = np.argmax(rec_lgbm[:-1] * f1_lgbm[:-1])
else:
    print("✅ Targets Reached on training folds.")
    valid_indices = np.where(mask_lgbm)[0]
    best_idx_lgbm = valid_indices[np.argmax(f1_lgbm[valid_indices])] # Highest F1 that keeps Recall high

optimal_threshold_lgbm = thresh_lgbm[best_idx_lgbm]

# Final evaluation
lgbm_proba_test = lgbm_best.predict_proba(X_test)[:, 1]
lgbm_preds_final = (lgbm_proba_test >= optimal_threshold_lgbm).astype(int)
test_proba["LightGBM"] = lgbm_proba_test # Save for the final comparison plots

print("\n" + "-"*55)
print(f" LIGHTGBM FINAL REPORT (Threshold: {optimal_threshold_lgbm:.4f})")
print("-"*55)
print(classification_report(y_test, lgbm_preds_final, digits=4))

# Economic Impact Check
prec_final = precision_score(y_test, lgbm_preds_final)
impact = 5 / prec_final if prec_final > 0 else float('inf')
print(f"Economic Impact: €{impact:.2f} | Target Met: {impact < 35}")

## Voting Classifier

In [ ]:
# Optuna objective
def objective_voting(trial):
    # Model A: XGBoost Parameters
    xgb_params = {
        "n_estimators": trial.suggest_int("xgb_n_estimators", 200, 800),
        "max_depth": trial.suggest_int("xgb_max_depth", 3, 7),
        "learning_rate": trial.suggest_float("xgb_learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("xgb_subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("xgb_colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("xgb_reg_alpha", 0.0001, 10.0, log=True),
        "reg_lambda": trial.suggest_float("xgb_reg_lambda", 0.0001, 10.0, log=True),
        "min_child_weight": trial.suggest_int("xgb_min_child_weight", 1, 10),
        "scale_pos_weight": trial.suggest_float("xgb_scale_pos_weight", 1.0, 10.0),
    }

    # Model B: LightGBM Parameters
    lgbm_params = {
        "n_estimators": trial.suggest_int("lgbm_n_estimators", 200, 800),
        "max_depth": trial.suggest_int("lgbm_max_depth", 3, 8),
        "learning_rate": trial.suggest_float("lgbm_learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("lgbm_num_leaves", 20, 100),
        "subsample": trial.suggest_float("lgbm_subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("lgbm_colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("lgbm_reg_alpha", 0.0001, 10.0, log=True),
        "reg_lambda": trial.suggest_float("lgbm_reg_lambda", 0.0001, 10.0, log=True),
        "class_weight": "balanced",
    }

    # Vote Weights
    w_xgb = trial.suggest_float("w_xgb", 0.1, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.1, 1.0)

    xgb_model = XGBClassifier(**xgb_params, random_state=random_state, eval_metric="logloss", verbosity=0)
    lgbm_model = LGBMClassifier(**lgbm_params, random_state=random_state, verbose=-1)

    voting = VotingClassifier(
        estimators=[("xgb", xgb_model), ("lgbm", lgbm_model)],
        voting="soft", weights=[w_xgb, w_lgbm]
    )

    pipeline = Pipeline([
        ("scaler/encoder", clone(preprocessor)),
        ("classifier", voting)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring = scoring, n_jobs=-1)
    return scores.mean() - (scores.std() * 0.5)

# Running the study
study_voting = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_state))
study_voting.optimize(objective_voting, n_trials=n_trials, show_progress_bar=True)

# Refit the best voters
p = study_voting.best_params
voting_best = VotingClassifier(
    estimators=[
        ("xgb", XGBClassifier(
            n_estimators=p["xgb_n_estimators"], max_depth=p["xgb_max_depth"], 
            learning_rate=p["xgb_learning_rate"], subsample=p["xgb_subsample"], 
            colsample_bytree=p["xgb_colsample_bytree"], reg_alpha=p["xgb_reg_alpha"], 
            reg_lambda=p["xgb_reg_lambda"], min_child_weight=p["xgb_min_child_weight"], 
            scale_pos_weight=p["xgb_scale_pos_weight"], random_state=random_state, eval_metric="logloss")),
        ("lgbm", LGBMClassifier(
            n_estimators=p["lgbm_n_estimators"], max_depth=p["lgbm_max_depth"], 
            learning_rate=p["lgbm_learning_rate"], num_leaves=p["lgbm_num_leaves"], 
            subsample=p["lgbm_subsample"], colsample_bytree=p["lgbm_colsample_bytree"], 
            reg_alpha=p["lgbm_reg_alpha"], reg_lambda=p["lgbm_reg_lambda"], 
            class_weight="balanced", random_state=random_state, verbose=-1))
    ],
    voting="soft", weights=[p["w_xgb"], p["w_lgbm"]]
)

voting_pipeline = Pipeline([("scaler/encoder", clone(preprocessor)), ("classifier", voting_best)])
voting_pipeline.fit(X_train, y_train)

voting_pred  = voting_pipeline.predict(X_test)
voting_proba = voting_pipeline.predict_proba(X_test)[:, 1]

test_proba["Voting Ensemble"] = voting_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, voting_pred, digits=2))
print("-"*55)

cost_per_call  = 5
precision_cl1  = classification_report(y_test, voting_pred, output_dict=True)['1']['precision']
impact   = cost_per_call / precision_cl1 if precision_cl1 > 0 else float('inf')
print(f"Economic Impact: €{impact:.2f} per subscriber")
print(f"Target Met: {impact < 35}")

In [ ]:
y_train_probas_vote = cross_val_predict(voting_pipeline, X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

# Calculate metrics
prec_v, rec_v, thresh_v = precision_recall_curve(y_train, y_train_probas_vote)
f1_v = (2 * prec_v * rec_v) / (prec_v + rec_v + 1e-9)

# Apply target metric constraints
mask_v = (rec_v[:-1] > 0.75) & (f1_v[:-1] > 0.55)

if not any(mask_v):
    print("❌ Targets Unreachable: Optimizing for best compromise...")
    best_idx_v = np.argmax(rec_v[:-1] * f1_v[:-1])
else:
    print("✅ Targets Reached on training folds.")
    valid_indices = np.where(mask_v)[0]
    best_idx_v = valid_indices[np.argmax(f1_v[valid_indices])]

optimal_threshold_v = thresh_v[best_idx_v]

# Final evaluation
voting_proba_test = voting_pipeline.predict_proba(X_test)[:, 1]
voting_preds_final = (voting_proba_test >= optimal_threshold_v).astype(int)

print("\n" + "="*55)
print(f" VOTING ENSEMBLE FINAL REPORT (Threshold: {optimal_threshold_v:.4f})")
print("="*55)
print(classification_report(y_test, voting_preds_final, digits=4))

# Economic Impact Check
prec_final = precision_score(y_test, voting_preds_final)
impact = 5 / prec_final if prec_final > 0 else float('inf')
print(f"Economic Impact: €{impact:.2f} | Target Met: {impact < 35}")

## ROC-AUC Chart
---

In [ ]:
# 1. Update Colors to include your final Ensemble
COLORS = {
    "Logistic Regression": "#4A90D9",
    "XGBoost": "#E74C3C",
    "LightGBM": "#F39C12",
    "Voting Ensemble": "#9B59B6"  # Added Purple for the final boss
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("ROC-AUC Comparison: Tuned Models vs Voting Ensemble", 
             fontsize=16, fontweight="bold", y=1.05)

# 2. LEFT: ROC Curves
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier", alpha=0.5)

rows = []
# Ensure we plot in order of performance for the legend
for name, proba in test_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    score = auc(fpr, tpr)
    rows.append({"Model": name, "AUC": score})
    
    sns.lineplot(x=fpr, y=tpr, ax=ax, lw=2.5, color=COLORS.get(name, "#333333"),
                 label=f"{name} (AUC = {score:.4f})")

ax.set_title("ROC Curves", fontsize=14, pad=10)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.grid(alpha=0.3)
ax.legend(loc="lower right", frameon=True)

# 3. RIGHT: AUC Bar Chart
ax2 = axes[1]
df_aucs = pd.DataFrame(rows).sort_values("AUC", ascending=False)

sns.barplot(data=df_aucs, x='AUC', y='Model', palette=COLORS, 
            ax=ax2, hue='Model', legend=False, alpha=0.9, edgecolor="white")

# Add white text labels inside the bars for readability
for i, val in enumerate(df_aucs['AUC']):
    ax2.text(val - 0.005, i, f"{val:.4f}", va='center', ha='right', 
             color='white', fontweight='bold', fontsize=11)

ax2.set_title("AUC Score Comparison (Leaderboard)", fontsize=14, pad=10)

# Set X-limit to 'zoom in' on the differences (usually between 0.8 and 1.0)
min_auc = df_aucs['AUC'].min()
ax2.set_xlim([max(0, min_auc - 0.05), 1.0])
ax2.set_xlabel("ROC-AUC Score")
ax2.set_ylabel("") 

plt.tight_layout()
plt.show()

# 4. Final Console Leaderboard
print("\n" + "═"*45)
print(f"{'RANK':<5} {'MODEL':<25} {'TEST AUC':<10}")
print("─" * 45)
for i, row in enumerate(df_aucs.itertuples(), 1):
    star = "⭐" if i == 1 else "  "
    print(f"{i:<5} {row.Model:<25} {row.AUC:<10.4f} {star}")
print("═"*45)

## Feature Importance
---

In [ ]:
# ── 1. Define the Plotting Function ──────────────────────────────────────────
def plot_feature_importance(importances, feature_names, title, color, top_n=20):
    df_imp = pd.DataFrame({"feature": feature_names, "importance": importances})
    # Sort by absolute value to handle LR coefficients and tree importances consistently
    df_imp = df_imp.reindex(df_imp["importance"].abs().sort_values(ascending=False).index)
    df_imp = df_imp.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(df_imp["feature"][::-1], df_imp["importance"][::-1], color=color)
    ax.set_title(title, fontsize=14, pad=15)
    ax.set_xlabel("Importance / Coefficient Value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

# ── 2. Prepare Feature Names ────────────────────────────────────────────────
feature_names = preprocessor.get_feature_names_out().tolist()

# ── 3. Logistic Regression ──────────────────────────────────────────────────
plot_feature_importance(
    importances   = lr_best.named_steps['classifier'].coef_[0],
    feature_names = feature_names,
    title         = "Logistic Regression — Feature Coefficients",
    color         = "#4A90D9"
)

# ── 4. XGBoost ──────────────────────────────────────────────────────────────
plot_feature_importance(
    importances   = xgb_best.named_steps['classifier'].feature_importances_,
    feature_names = feature_names,
    title         = "XGBoost — Feature Importance",
    color         = "#E74C3C"
)

# ── 5. LightGBM ─────────────────────────────────────────────────────────────
plot_feature_importance(
    importances   = lgbm_best.named_steps['classifier'].feature_importances_,
    feature_names = feature_names,
    title         = "LightGBM — Feature Importance",
    color         = "#F39C12"
)

# ── 6. Voting Classifier (Weighted Average) ──────────────────────────────────
# Extract weights and underlying models
v_clf = voting_pipeline.named_steps['classifier']
weights = v_clf.weights

# Extract normalized importances from the ensemble members
# We normalize to ensure XGB and LGBM are on the same 0-1 scale before weighting
xgb_imp_norm = v_clf.named_estimators_['xgb'].feature_importances_ / v_clf.named_estimators_['xgb'].feature_importances_.sum()
lgbm_imp_norm = v_clf.named_estimators_['lgbm'].feature_importances_ / v_clf.named_estimators_['lgbm'].feature_importances_.sum()

# Compute weighted average importance
voting_importance = (weights[0] * xgb_imp_norm + weights[1] * lgbm_imp_norm) / sum(weights)

plot_feature_importance(
    importances   = voting_importance,
    feature_names = feature_names,
    title         = "Voting Ensemble — Weighted Feature Importance",
    color         = "#9B59B6" # Purple
)